In [13]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, classification_report
)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# Load data and create dfs
canine_df = pd.read_pickle('/content/drive/MyDrive/210 Capstone/final_data/canine_df_split.pkl')
human_df  = pd.read_pickle('/content/drive/MyDrive/210 Capstone/final_data/human_df.pkl')

print("Top 5 rows of canine_df:")
display(canine_df.head())

print("\nTop 5 rows of human_df:")
display(human_df.head())

Top 5 rows of canine_df:


,file_identifier,timestamp,glucose,hypoglycemia,hyperglycemia,glucose_category,dt_min,roc,safe
0,canine1,2025-11-01 05:51:00,375.0,False,True,Hyperglycemia,NaN,NaN,False
1,canine1,2025-11-01 06:06:00,375.0,False,True,Hyperglycemia,15.0,0.000000,False
2,canine1,2025-11-01 06:21:00,373.0,False,True,Hyperglycemia,15.0,-0.133333,False
3,canine1,2025-11-01 06:36:00,375.0,False,True,Hyperglycemia,15.0,0.133333,False
4,canine1,2025-11-01 06:51:00,376.0,False,True,Hyperglycemia,15.0,0.066667,False



Top 5 rows of human_df:


,file_identifier,timestamp,glucose,hypoglycemia,hyperglycemia,glucose_category,dt_min,roc,safe
142125,Patient_109,2001-04-27 00:00:25,179,False,False,Normal,NaN,NaN,True
142126,Patient_109,2001-04-27 00:10:31,189,False,True,Hyperglycemia,10.100000,0.990099,False
142127,Patient_109,2001-04-27 00:20:35,191,False,True,Hyperglycemia,10.066667,0.198675,False
142128,Patient_109,2001-04-27 00:30:34,193,False,True,Hyperglycemia,9.983333,0.200334,False
142129,Patient_109,2001-04-27 00:40:28,196,False,True,Hyperglycemia,9.900000,0.303030,False


In [14]:
print("Column Information for canine_df:")
display(canine_df.info())

print("\nDescriptive Statistics for canine_df:")
display(canine_df.describe())

Column Information for canine_df:
<class 'pandas.core.frame.DataFrame'>
Index: 31318 entries, 0 to 20859
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   file_identifier   31318 non-null  object        
 1   timestamp         31318 non-null  datetime64[ns]
 2   glucose           31318 non-null  float64       
 3   hypoglycemia      31318 non-null  bool          
 4   hyperglycemia     31318 non-null  bool          
 5   glucose_category  31318 non-null  object        
 6   dt_min            31308 non-null  float64       
 7   roc               30350 non-null  float64       
 8   safe              31318 non-null  bool          
dtypes: bool(3), datetime64[ns](1), float64(3), object(2)
memory usage: 1.8+ MB


None


Descriptive Statistics for canine_df:


,timestamp,glucose,dt_min,roc
count,31318,31318.000000,31308.000000,30350.000000
mean,2025-09-04 18:47:00.434893312,303.328661,14.446499,0.045975
min,2025-03-22 17:43:00,40.000000,0.000000,-74.000000
25%,2025-04-19 02:25:00,193.000000,5.000000,-0.800000
50%,2025-10-26 12:50:00,304.000000,15.000000,0.000000
75%,2025-12-05 02:51:15,415.000000,15.000000,0.800000
max,2026-02-04 12:57:00,500.000000,12641.000000,89.000000
std,NaN,130.590247,94.501400,3.271692


**BASELINE MODEL 1-1: Current-State Baseline (safe now)**

Is the dataset coherent?


In [15]:
def prepare_xy(df: pd.DataFrame):
    df = df.copy()
    df = df.sort_values(["file_identifier", "timestamp"])

    # Target stays boolean
    y = df["safe"]  # bool

    # Features (avoid label leakage)
    X = df[["glucose", "dt_min", "roc"]].copy()
    X["hour"] = df["timestamp"].dt.hour
    X["dow"] = df["timestamp"].dt.dayofweek

    groups = df["file_identifier"].astype(str)
    return X, y, groups

def train_eval_logreg(df: pd.DataFrame, test_size=0.2, random_state=42):
    X, y, groups = prepare_xy(df)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), X.columns.tolist()),
        ],
        remainder="drop"
    )

    model = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    pred = (proba >= 0.5)

    # Convert to int just for metrics / display consistency
    y_test_int = y.iloc[test_idx].astype(int)
    pred_int = pred.astype(int)

    print("ROC-AUC:", roc_auc_score(y_test_int, proba))
    print("PR-AUC :", average_precision_score(y_test_int, proba))
    print("F1     :", f1_score(y_test_int, pred_int))
    print("Confusion matrix:\n", confusion_matrix(y_test_int, pred_int))
    print("\nClassification report:\n", classification_report(y_test_int, pred_int, digits=3))

    return model



In [16]:

canine_model = train_eval_logreg(canine_df)
human_model  = train_eval_logreg(human_df)


ROC-AUC: 0.9790382696484122
PR-AUC : 0.8577070213191815
F1     : 0.9632107023411371
Confusion matrix:
 [[1827   66]
 [   0  864]]

Classification report:
               precision    recall  f1-score   support

           0      1.000     0.965     0.982      1893
           1      0.929     1.000     0.963       864

    accuracy                          0.976      2757
   macro avg      0.965     0.983     0.973      2757
weighted avg      0.978     0.976     0.976      2757

ROC-AUC: 0.9640992666655909
PR-AUC : 0.9183431932386007
F1     : 0.9336979258099933
Confusion matrix:
 [[ 99364   3688]
 [ 18595 156900]]

Classification report:
               precision    recall  f1-score   support

           0      0.842     0.964     0.899    103052
           1      0.977     0.894     0.934    175495

    accuracy                          0.920    278547
   macro avg      0.910     0.929     0.916    278547
weighted avg      0.927     0.920     0.921    278547



**Result Interpretation:**

Baseline logistic regression (current-state safe/unsafe):
On both canine and human datasets, the current-state model achieves strong discrimination (ROC-AUC ≈ 0.96–0.98), indicating that the selected real-time features (glucose, roc, dt_min, time features) carry substantial signal for distinguishing safe vs unsafe states.

Canine results show near-perfect recall for the “safe” class with low false positives, suggesting the canine data is highly separable under this feature set.

Human results remain strong but show a more conservative classification pattern: precision for “safe” is very high, while recall is lower, meaning the model tends to label some truly safe points as unsafe.

**Overall**, this baseline validates the end-to-end modeling pipeline and provides a strong reference point before moving to the more clinically relevant early-warning prediction task.


**BASELINE MODEL 1-2: Early Warning Model (predict safe at +H minutes)**


In [18]:
def add_future_safe_label(df, horizon_min: int, dt_col="dt_min"):
    """
    Create y_future = safe shifted forward by k steps, where
    k ≈ horizon_min / median(dt_min) within each file_identifier.

    Assumes timestamp already datetime64 and data is roughly regular cadence (e.g., 15 min).
    """
    df = df.copy()
    df = df.sort_values(["file_identifier", "timestamp"])

    def _per_file(g):
        step = g[dt_col].median()  # e.g., ~15
        if np.isnan(step) or step <= 0:
            g["y_future"] = np.nan
            return g
        k = int(round(horizon_min / step))
        g["y_future"] = g["safe"].shift(-k)
        return g

    df = df.groupby("file_identifier", group_keys=False).apply(_per_file)
    df = df.dropna(subset=["y_future"])
    df["y_future"] = df["y_future"].astype(bool)
    return df

def prepare_xy_early_warning(df, horizon_min: int):
    df2 = add_future_safe_label(df, horizon_min=horizon_min)

    # Same “real-time” features as before
    X = df2[["glucose", "dt_min", "roc"]].copy()
    X["hour"] = df2["timestamp"].dt.hour
    X["dow"]  = df2["timestamp"].dt.dayofweek

    y = df2["y_future"]               # bool target: safe at +H
    groups = df2["file_identifier"].astype(str)
    return X, y, groups

def train_eval_logreg_early_warning(df, horizon_min=30, test_size=0.2, random_state=42):
    X, y, groups = prepare_xy_early_warning(df, horizon_min=horizon_min)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), X.columns.tolist()),
        ],
        remainder="drop"
    )

    model = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    pred  = (proba >= 0.5)

    y_test_int = y.iloc[test_idx].astype(int)
    pred_int   = pred.astype(int)

    print(f"=== Early warning: predict safe at +{horizon_min} min ===")
    print("ROC-AUC:", roc_auc_score(y_test_int, proba))
    print("PR-AUC :", average_precision_score(y_test_int, proba))
    print("F1     :", f1_score(y_test_int, pred_int))
    print("Confusion matrix:\n", confusion_matrix(y_test_int, pred_int))
    print("\nClassification report:\n", classification_report(y_test_int, pred_int, digits=3))

    return model


In [19]:
canine_ew30 = train_eval_logreg_early_warning(canine_df, horizon_min=30)
human_ew30  = train_eval_logreg_early_warning(human_df,  horizon_min=30)


/tmp/ipython-input-4020123011.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("file_identifier", group_keys=False).apply(_per_file)


=== Early warning: predict safe at +30 min ===
ROC-AUC: 0.9619060398407937
PR-AUC : 0.8455659670292874
F1     : 0.8794326241134752
Confusion matrix:
 [[1726  163]
 [  58  806]]

Classification report:
               precision    recall  f1-score   support

           0      0.967     0.914     0.940      1889
           1      0.832     0.933     0.879       864

    accuracy                          0.920      2753
   macro avg      0.900     0.923     0.910      2753
weighted avg      0.925     0.920     0.921      2753



/tmp/ipython-input-4020123011.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("file_identifier", group_keys=False).apply(_per_file)


=== Early warning: predict safe at +30 min ===
ROC-AUC: 0.9247709971443248
PR-AUC : 0.8995333496800785
F1     : 0.8879100124318638
Confusion matrix:
 [[ 92430  10592]
 [ 26916 148558]]

Classification report:
               precision    recall  f1-score   support

           0      0.774     0.897     0.831    103022
           1      0.933     0.847     0.888    175474

    accuracy                          0.865    278496
   macro avg      0.854     0.872     0.860    278496
weighted avg      0.875     0.865     0.867    278496



**Result Interpretation:**

Early-warning (+30 min) logistic regression:
We reframed the task from predicting the current state to predicting whether the subject will be safe in 30 minutes using only information available at the current timestamp (glucose, roc, dt_min, time features).

As expected, performance decreases compared with the current-state baseline because future state is harder to predict. However, the model still achieves strong discrimination (ROC-AUC ≈ 0.92–0.96, PR-AUC ≈ 0.85–0.90).

The results show a practical tradeoff between missed events vs false alarms. In one dataset the model is slightly more “optimistic” (more future-safe predictions, higher recall for safe but lower precision), while in the other it is more conservative (higher precision for safe but lower recall). This suggests threshold tuning can align the model behavior with the risk preference (e.g., minimizing predictions of “safe” when the future state is unsafe).

**BASELINE MODEL 1-3: Early Warning Model (predict unsafe within +H minutes)**

In [20]:
# Early Warning Model: predict UNSAFE within next +H minutes (irregular 10–15 min cadence supported)

# -----------------------------
# 1) Label creation (time-based window, works with irregular cadence)
# -----------------------------
def add_unsafe_within_time_window_label(
    df: pd.DataFrame,
    horizon_min: int = 30,
    group_col: str = "file_identifier",
    time_col: str = "timestamp",
    safe_col: str = "safe",
    label_col: str = "unsafe_within"
) -> pd.DataFrame:
    """
    Create label: unsafe_within = True if ANY future point within (t, t+horizon] is unsafe.
    If there is no future point within the horizon window, label is set to NA and dropped later.
    """
    df = df.copy()
    df = df.sort_values([group_col, time_col])

    def _per_group(g: pd.DataFrame) -> pd.DataFrame:
        t = g[time_col].to_numpy(dtype="datetime64[ns]")
        unsafe = (~g[safe_col].to_numpy(dtype=bool)).astype(np.int32)  # 1 if unsafe else 0

        prefix = np.concatenate([[0], np.cumsum(unsafe)])  # length n+1

        t_end = t + np.timedelta64(horizon_min, "m")
        right = np.searchsorted(t, t_end, side="right")  # first index > t_end

        left = np.arange(len(t)) + 1  # exclude current point
        has_future = right > left

        unsafe_count = prefix[right] - prefix[left]

        out = np.full(len(t), np.nan, dtype=float)
        out[has_future] = (unsafe_count[has_future] > 0).astype(float)

        g[label_col] = pd.Series(out, index=g.index).astype("boolean")
        return g

    df = df.groupby(group_col, group_keys=False).apply(_per_group)
    df = df.dropna(subset=[label_col])
    df[label_col] = df[label_col].astype(bool)
    return df


# -----------------------------
# 2) Feature preparation
# -----------------------------
def prepare_xy_unsafe_within(df: pd.DataFrame, horizon_min: int = 30):
    df2 = add_unsafe_within_time_window_label(df, horizon_min=horizon_min)

    X = df2[["glucose", "dt_min", "roc"]].copy()
    X["hour"] = df2["timestamp"].dt.hour
    X["dow"] = df2["timestamp"].dt.dayofweek

    y = df2["unsafe_within"]  # True = will be unsafe within next H minutes
    groups = df2["file_identifier"].astype(str)
    return X, y, groups


# -----------------------------
# 3) Helper: print model parameters (after fitting)
# -----------------------------
def print_logreg_parameters(fitted_pipeline: Pipeline, feature_names):
    """
    Prints intercept and coefficients of the LogisticRegression inside the pipeline.
    Note: coefficients correspond to STANDARDIZED features (because we use StandardScaler).
    """
    clf = fitted_pipeline.named_steps["clf"]
    coefs = clf.coef_.ravel()
    intercept = float(clf.intercept_[0])

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coef": coefs,
        "abs_coef": np.abs(coefs),
    }).sort_values("abs_coef", ascending=False)

    print("\n--- Logistic Regression Parameters (standardized feature space) ---")
    print("Intercept:", intercept)
    print(coef_df.to_string(index=False))


# -----------------------------
# 4) Train + evaluate logistic regression (and print params)
# -----------------------------
def train_eval_logreg_unsafe_within(
    df: pd.DataFrame,
    horizon_min: int = 30,
    test_size: float = 0.2,
    random_state: int = 42,
    threshold: float = 0.5,
    print_params: bool = True
):
    X, y, groups = prepare_xy_unsafe_within(df, horizon_min=horizon_min)
    feature_names = X.columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), feature_names),
        ],
        remainder="drop"
    )

    model = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)

    # Print learned parameters (after fit)
    if print_params:
        print_logreg_parameters(model, feature_names)

    # Probabilities: class 1 corresponds to True (unsafe within window)
    proba_unsafe = model.predict_proba(X_test)[:, 1]
    pred_unsafe = (proba_unsafe >= threshold)

    y_test_int = y_test.astype(int)
    pred_int = pred_unsafe.astype(int)

    print(f"\n=== Early warning: predict UNSAFE within +{horizon_min} min (threshold={threshold}) ===")
    print("ROC-AUC:", roc_auc_score(y_test_int, proba_unsafe))
    print("PR-AUC :", average_precision_score(y_test_int, proba_unsafe))
    print("F1     :", f1_score(y_test_int, pred_int))
    print("Confusion matrix:\n", confusion_matrix(y_test_int, pred_int))
    print("\nClassification report:\n", classification_report(y_test_int, pred_int, digits=3))

    return model



In [22]:
canine_labeled = add_unsafe_within_time_window_label(canine_df, horizon_min=30)
canine_labeled.head()

/tmp/ipython-input-109682421.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(group_col, group_keys=False).apply(_per_group)


,file_identifier,timestamp,glucose,hypoglycemia,hyperglycemia,glucose_category,dt_min,roc,safe,unsafe_within
0,canine1,2025-11-01 05:51:00,375.0,False,True,Hyperglycemia,NaN,NaN,False,True
1,canine1,2025-11-01 06:06:00,375.0,False,True,Hyperglycemia,15.0,0.000000,False,True
2,canine1,2025-11-01 06:21:00,373.0,False,True,Hyperglycemia,15.0,-0.133333,False,True
3,canine1,2025-11-01 06:36:00,375.0,False,True,Hyperglycemia,15.0,0.133333,False,True
4,canine1,2025-11-01 06:51:00,376.0,False,True,Hyperglycemia,15.0,0.066667,False,True


In [21]:
# -----------------------------
# 5) Usage
# -----------------------------
canine_model_unsafe30 = train_eval_logreg_unsafe_within(canine_df, horizon_min=30, threshold=0.5)
human_model_unsafe30  = train_eval_logreg_unsafe_within(human_df,  horizon_min=30, threshold=0.5)


/tmp/ipython-input-109682421.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(group_col, group_keys=False).apply(_per_group)



--- Logistic Regression Parameters (standardized feature space) ---
Intercept: 1.7753015942364472
feature      coef  abs_coef
glucose  4.382966  4.382966
 dt_min -0.215233  0.215233
    roc  0.111370  0.111370
   hour  0.097804  0.097804
    dow  0.050514  0.050514

=== Early warning: predict UNSAFE within +30 min (threshold=0.5) ===
ROC-AUC: 0.9632606710607143
PR-AUC : 0.9882122216734046
F1     : 0.946452123450277
Confusion matrix:
 [[ 721   54]
 [ 149 1794]]

Classification report:
               precision    recall  f1-score   support

           0      0.829     0.930     0.877       775
           1      0.971     0.923     0.946      1943

    accuracy                          0.925      2718
   macro avg      0.900     0.927     0.912      2718
weighted avg      0.930     0.925     0.927      2718



/tmp/ipython-input-109682421.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(group_col, group_keys=False).apply(_per_group)



--- Logistic Regression Parameters (standardized feature space) ---
Intercept: 0.035521008327876656
feature     coef  abs_coef
glucose 1.866093  1.866093
    roc 0.170465  0.170465
   hour 0.023058  0.023058
    dow 0.007089  0.007089
 dt_min 0.004676  0.004676

=== Early warning: predict UNSAFE within +30 min (threshold=0.5) ===
ROC-AUC: 0.934156388739078
PR-AUC : 0.951876590919171
F1     : 0.8641460953299587
Confusion matrix:
 [[137414  23061]
 [ 10460 106611]]

Classification report:
               precision    recall  f1-score   support

           0      0.929     0.856     0.891    160475
           1      0.822     0.911     0.864    117071

    accuracy                          0.879    277546
   macro avg      0.876     0.883     0.878    277546
weighted avg      0.884     0.879     0.880    277546



**Early-warning (unsafe within 30 minutes) baseline – canine:**

Using a timestamp-based window label, the logistic regression achieves strong discrimination (ROC-AUC = 0.963, PR-AUC = 0.988). At the default threshold 0.5, the model reaches precision = 0.971 and recall = 0.923 for predicting “unsafe within 30 minutes,” indicating few false alarms and relatively few missed unsafe windows.

**What this means operationally**
* **Recall** for unsafe-within (class 1) = 0.923: The model catched about 92.3% of unsafe-within-30min windows.
* **Precision** for unsafe-within = 0.971: About 97.1% of the time when alert is right.

So the baseline model is both:
* high-precision (few nuisance texts)
* high-recall (few missed unsafe windows)

This provides a strong baseline for comparison against more advanced time-series models.